# Capstone Project: Cease & Desist Document Processing System

Enterprises receive **Cease & Desist** requests from customers who want to stop all direct communication. Currently, human agents must manually read scanned PDF documents to determine if each request is legitimate, which is:
- ⏱️ Time-consuming and slow
- 💰 Expensive (requires human review)
- ❌ Error-prone (human fatigue, inconsistency)
- 📈 Not scalable (volume increases over time)


### Mission

Build an **intelligent multi-agent system** that automates the classification and processing of Cease & Desist documents, reducing manual effort while maintaining accuracy and compliance.

## 📋 Solution Requirements

### Core Functionality

Your system must:

1. **Classify Documents** into 3 categories:
   - ✅ **"Cease"** - Valid cease & desist request
   - ⚠️ **"Uncertain"** - Requires manual review
   - ❌ **"Irrelevant"** - Not a cease request

2. **Process Based on Classification:**
   - **Cease Requests** → Call database agent to store:
     - Date of document received
     - Document name
     - Extracted details
   
   - **Irrelevant Documents** → Call archiving agent to write to flat file:
     - Date of document received
     - Document name
   
   - **Uncertain Cases** → Present to human agent for review (HITL)

3. **Audit Everything:**
   - Log all requests with explanations
   - Track classification decisions
   - Maintain compliance trail

## Env Setup

In [1]:
print("Installing dependencies...")
!pip install -qU langchain langchain-groq langgraph langchain-community chromadb sentence-transformers groq langsmith python-dotenv pypdf langchain_huggingface opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp-proto-grpc opentelemetry-exporter-otlp-proto-http --upgrade
print("ALL DEPENDENCIES INSTALLED")

Installing dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.1/168.1 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 359.9/359.9 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

## Load the GROQ API key from Colab userdata into the environment.

In [3]:
# 2. Setup API Keys
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
# Optional for LangSmith Tracking
os.environ["LANGSMITH_API_KEY"] = userdata.get('LANGSMITH_API_KEY')
os.environ["LANGSMITH_TRACING_V2"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"] = "Cease & Desist - Capstone Project"

print("\n API Keys are set")


 API Keys are set


## Initialize the Groq LLM with structured output

In [4]:
# 3. Initialize the Groq LLM
from langchain.agents import create_agent
from langchain_groq import ChatGroq
from pydantic import BaseModel, Field

class ClassificationOutput(BaseModel):
    classification: str
    confidence: float
    reasoning: str
    extracted_details: str

#Initilize llm
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    max_tokens=800,
)

print("\n LLM Initiated...")


 LLM Initiated...


## Load files from Git hub repo
Repo : https://github.com/jayyanar/agentic-ai-training

Files_Path : day5/capstone-project/data/pdfs

Branch : main

Fetch PDF files from GitHub using raw URLs, temporarily store them, load them via PyPDFLoader into LangChain Document objects, and attach metadata for traceability before passing them into the AI pipeline.


GitHub Repo → Get file names → Build RAW URL → Download file → Save temp → Load with PyPDFLoader → Add metadata → Ready for pipeline

In [5]:
# 4. Load files from Git Hub repo
import requests
from tempfile import NamedTemporaryFile
from langchain_community.document_loaders import PyPDFLoader

# step 1: Get the file names into and array
API_URL = "https://api.github.com/repos/jayyanar/agentic-ai-training/contents/day5/capstone-project/data/pdfs"

response = requests.get(API_URL)
files_data = response.json()

files = [file["name"] for file in files_data]

print(files)

# step 2: Using the base URL, get the files downloaded

BASE_RAW = "https://raw.githubusercontent.com/jayyanar/agentic-ai-training/main/day5/capstone-project/data/pdfs/"

documents = []

for file in files:

    url = BASE_RAW + file   # convert to raw file URL

    res = requests.get(url)

    if res.status_code != 200: #Print the error message if the file/path couldnt not be located and continue the loop
        print(f"Failed: {file}")
        continue

    # Save as temp file
    with NamedTemporaryFile(delete=False) as tmp:
        tmp.write(res.content)
        temp_path = tmp.name

    # Load PDF
    loader = PyPDFLoader(temp_path)
    docs = loader.load()
    print(f"Loaded: {file}")

    # Add metadata - This metadata used later to identify the files inorder to classify and insert into DB or Excel
    for d in docs:
        d.metadata["source"] = file

    documents.extend(docs)


['01_copyright_infringement_photography.pdf', '02_trademark_infringement_tech.pdf', '03_trade_secret_misappropriation.pdf', '04_defamation_online_review.pdf', '05_patent_infringement_medical_device.pdf', '06_harassment_workplace.pdf', '07_software_license_violation.pdf', '08_non_compete_violation.pdf', '09_copyright_infringement_music.pdf', '10_breach_of_contract_nda.pdf', 'LOA2.pdf', 'LOA3.pdf', 'LOA4.pdf', 'LOA5.pdf', 'LOA6.pdf', 'LOA7.pdf', 'LOA8.pdf', 'LOA9.pdf', 'LoA1.pdf', 'bw_doc_1.pdf', 'bw_doc_2.pdf', 'bw_doc_3.pdf', 'bw_doc_4.pdf', 'bw_doc_5.pdf', 'notice_1.pdf', 'notice_2.pdf', 'notice_3.pdf', 'notice_4.pdf', 'notice_5.pdf']
Loaded: 01_copyright_infringement_photography.pdf
Loaded: 02_trademark_infringement_tech.pdf
Loaded: 03_trade_secret_misappropriation.pdf
Loaded: 04_defamation_online_review.pdf
Loaded: 05_patent_infringement_medical_device.pdf
Loaded: 06_harassment_workplace.pdf
Loaded: 07_software_license_violation.pdf
Loaded: 08_non_compete_violation.pdf
Loaded: 09_co

## Split the Documents, Embedding using HuggingFace and store in Vector DB

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

print("Split documents with chunk size 500 and overlap of 50...")
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

splits = splitter.split_documents(documents)
print("Split documents complete...")

# Embed & Store
print("Creating vector store...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)
retriever = vectorstore.as_retriever()
print("Vector store created.")



Split documents with chunk size 500 and overlap of 50...
Split documents complete...
Creating vector store...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector store created.


## Set-Up Database


In [7]:
import sqlite3
import json

from datetime import datetime

conn = sqlite3.connect("Sachin_capstone_cease_desist.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS cease_requests (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    document_name TEXT,
    date_received TEXT,
    extracted_details TEXT,
    classification TEXT,
    confidence REAL,
    reasoning TEXT
)
""")

conn.commit()

print("DATABASE Created : Sachin_capstone_cease_desist.db")

def log_audit(data):
    with open("Sachin_capstone_cease_desist_audit_log.json", "a") as f:
        f.write(json.dumps(data) + "\n")

print("Log is  Created : Sachin_capstone_cease_desist_audit_log.json")


DATABASE Created : Sachin_capstone_cease_desist.db
Log is  Created : Sachin_capstone_cease_desist_audit_log.json


## Detect scanned PDFs
If the texts are not extractable then the document is considered to be scanned.
Check if length of the extracted file, if it's minimal then its detected as scanned document and will be sent to Human for the external review.


In [8]:
def is_scanned_pdf(text):
    return len(text.strip()) < 50

## Classifier Agent - Main part of the Project
Classifier agent check the PDF documents, retrives the context and sends it to LLM. it classifies the documents, confidence level and reason for the classification

In [9]:
import json
import re

def extract_json(text):
    try:
        json_str = re.search(r"\{.*\}", text, re.DOTALL).group()
        return json.loads(json_str)
    except Exception as e:
        return {
            "classification": "UNCERTAIN",
            "confidence": 50,
            "reasoning": "JSON parsing failed",
            "extracted_details": text[:200]
        }


def classify_agent(state):

    document = state["document"]

# Detect scanned PDF
    if is_scanned_pdf(document):
        return {
            "document_name": state["document_name"],
            "classification": "UNCERTAIN",
            "confidence": 0,
            "reasoning": "Scanned document - Human review required",
            "extracted_details": "N/A",
            "scanned_document": True,
            "hitl_reason": "Scanned Document"
        }

    # RAG context
    context_docs = retriever.invoke(document)
    context = "\n".join([d.page_content for d in context_docs])


    prompt = f"""
You are an AI Legal Document Analysis Agent specialized in classifying documents into three categories: CEASE, IRRELEVANT and UNCERTAIN.

Rules:
- Return ONLY valid JSON
- No extra text
- Keep reasoning under 100 words
- Keep extracted_details very short (max 2 sentences)

IMPORTANT:
- Do NOT assume every document is a cease notice
- Only classify as CEASE if there is an explicit legal demand to stop an action
- If unsure, choose UNCERTAIN

Format:
{{
"classification": "CEASE | IRRELEVANT | UNCERTAIN",
"confidence": 1-100,
"reasoning": "short explanation",
"extracted_details": "brief summary"
}}

DOCUMENT:
{document[:3000]}

CONTEXT:
{context[:1500]}
"""

    response = llm.invoke(prompt)
    response_content = response.content

    parsed = extract_json(response_content)

    # Safe extraction
    classification = parsed.get("classification", "UNCERTAIN")
    classification = classification.strip().upper()
    confidence = int(parsed.get("confidence", 50))
    reasoning = parsed.get("reasoning", "No reasoning provided")
    extracted_details = parsed.get("extracted_details", "N/A")

    # Clamp confidence
    confidence = max(1, min(100, confidence))

    # Print to debug
    print("\n--- RAW RESPONSE ---\n", response_content)
    print("\n--- PARSED ---\n", parsed)

    # Audit log
    log_audit({
        "document": state["document_name"],
        "classification": classification,
        "confidence": confidence,
        "reasoning": reasoning
    })

    return {
        "document_name": state["document_name"],
        "classification": classification,
        "confidence": confidence,
        "reasoning": reasoning,
        "extracted_details": extracted_details,
        "scanned_document": False,
        "hitl_reason": None,
        "llm_classification": classification,
        "llm_confidence": confidence
    }


## Routing Logic
This agent decides where to store the classified document details.

1. if the document is classified as CEASE Store it in DB, If its IRRELEVANT store it an file, if UNCERTAIN then send it for Human Review  
2. if the human decision is CEASE then store in DB, if IRRELEVANT then store in file
3. if the confidence level is <70 then route to Human in review

In [10]:
def route_document(state):

    if "human_decision" in state:
        if state["human_decision"] == "CEASE":
            return "cease"
        elif state["human_decision"] == "IRRELEVANT":
            return "archive"
        else:
            return "end"

    if state["scanned_document"]:
        return "human"

    if state["confidence"] < 85:
        state["hitl_reason"] = "LOW_CONFIDENCE"
        return "human"

    if state["classification"] == "CEASE":
        return "cease"
    elif state["classification"] == "IRRELEVANT":
        return "archive"
    else:
        return "human"

## DB Agent
Insert into Database table for all the documents classified as CEASE

In [11]:
def db_agent(state):

    cursor.execute(
        """
        INSERT INTO cease_requests
        (document_name, date_received, extracted_details, classification, confidence, reasoning)
        VALUES (?, ?, ?, ?, ?, ?)
        """,
        (
            state["document_name"],
            str(datetime.now()),
            state["extracted_details"],
            state.get("final_classification", state["classification"]),
            state["confidence"],
            state["reasoning"]
        )
    )

    conn.commit()
    print("Stored in DB:", state["document_name"])
    return {}

## Archive Agent
Create a file for all the documents classified as IRRELEVANT

In [12]:
def archive_agent(state):

    with open("Irrelevant_Documents.txt", "a") as f:
        f.write(
            f"{datetime.now()} | {state.get('document_name')} | "
            f"Final: {state.get('human_decision', state.get('classification'))} | "
            f"LLM: {state.get('llm_classification')} | "
            f"Confidence: {state.get('llm_confidence', state.get('confidence'))} | "
            f"HITL Reason: {state.get('hitl_reason')} | "
            f"Reason: {state.get('reasoning')}\n"
        )

    print("Entered into File :", state["document_name"])
    return {}

## Human Agent

In [13]:
def human_agent(state):

    print("\n⚠️ HUMAN REVIEW REQUIRED")
    print("📄 Document:", state["document_name"])
    print("Scanned Document:", state.get("scanned_document"))
    print("LLM Classification:", state.get("classification"))
    print("Confidence:", state.get("confidence"))

    if state.get("hitl_reason") == "Scanned Document":
        print("Reason: Scanned Document")
    elif state.get("hitl_reason") == "LOW_CONFIDENCE":
        print("Reason: Low Confidence")

    user_input = input("Enter your choice options 1 (CEASE) or 2 (IRRELEVANT) : ").strip()

    mapping = {
        "1": "CEASE",
        "2": "IRRELEVANT"
    }

    human_decision = mapping.get(user_input, "IRRELEVANT")

    return {
        "document_name": state["document_name"],
        "classification": human_decision,   # For routing
        "human_decision": human_decision,
        "llm_confidence": state.get("llm_confidence"),
        "llm_classification": state.get("llm_classification"),
        "hitl_reason": state.get("hitl_reason"),
        "confidence": state.get("confidence"),
        "reasoning": "Human reviewed decision",
        "extracted_details": "Reviewed manually",
        "scanned_document": False
    }

## LangGraph Workflow

In [14]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

workflow = StateGraph(dict)

workflow.add_node("classifier", classify_agent)
workflow.add_node("cease", db_agent)
workflow.add_node("archive", archive_agent)
workflow.add_node("human", human_agent)

workflow.set_entry_point("classifier")

workflow.add_conditional_edges(
    "classifier",
    route_document,
    {"cease": "cease", "archive": "archive", "human": "human"}
)

workflow.add_conditional_edges(
    "human",
    route_document,
    {"cease": "cease", "archive": "archive", "end": END}
)

workflow.add_edge("cease", END)
workflow.add_edge("archive", END)

graph = workflow.compile(checkpointer=MemorySaver())

## Inturrupt for Human Input

In [15]:
from collections import defaultdict

grouped_docs = defaultdict(list)

# Group pages by file name
for doc in documents:
    grouped_docs[doc.metadata["source"]].append(doc.page_content)

# Process each file once
for file_name, pages in grouped_docs.items():

    full_text = "\n".join(pages)

    state = {
        "document": full_text,
        "document_name": file_name
    }

    print(f"\n --------------------------------------- Processing: {file_name} --------------------------------------------------")

    # Add a config for the checkpointer
    config = {"configurable": {"thread_id": file_name}}
    result = graph.invoke(state, config=config)

    # Handle HITL
    while "__interrupt__" in result:
        interrupt_data = result["__interrupt__"]

        print("\n HUMAN INPUT REQUIRED to Classify the Documents")
        print(interrupt_data)

        choice = input("Enter you option : '1' to clasify as CEASE, '2' to clasify as IRRELEVANT : ")

        # When resuming from interrupt, pass the same config
        result = graph.invoke({"choice": choice}, config=config)

    print("\n Completed:", file_name)


 --------------------------------------- Processing: 01_copyright_infringement_photography.pdf --------------------------------------------------

--- RAW RESPONSE ---
 {
"classification": "CEASE",
"confidence": 100,
"reasoning": "Explicit demand to stop unauthorized use of photographs with threat of legal action.",
"extracted_details": "Cease and desist letter for copyright infringement, demanding removal of photographs and payment of statutory damages."
}

--- PARSED ---
 {'classification': 'CEASE', 'confidence': 100, 'reasoning': 'Explicit demand to stop unauthorized use of photographs with threat of legal action.', 'extracted_details': 'Cease and desist letter for copyright infringement, demanding removal of photographs and payment of statutory damages.'}
Stored in DB: 01_copyright_infringement_photography.pdf

 Completed: 01_copyright_infringement_photography.pdf

 --------------------------------------- Processing: 02_trademark_infringement_tech.pdf -----------------------------

## Export the output to Excel file

In [16]:
import pandas as pd

df = pd.read_sql_query("SELECT * FROM cease_requests", conn)
df.to_excel("sachin_cease_requests.xlsx", index=False)

print("Excel exported!")

Excel exported!
